In [1]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types
from PIL import Image
import base64
import io
import json
from datetime import datetime
import uuid
import time
import re

import brand_director as bd
import nametag_prompts as pn

bd.setup_environment()

# 함수에서 반환된 키를 변수에 저장하여 다른 곳에서 활용
GEMINI_API_KEY, GEMINI_IMAGE_API_KEY = bd.load_gemini_keys()

# 시스템 프롬프트를 변수에 저장하여 다른 곳에서 활용
SYSTEM_PROMPT = pn.SYSTEM_PROMPT

✅ Gemini API 키 로드 완료
✅ Gemini Image API 키 로드 완료


In [ ]:
brand_info = {'brand_name': '숲결',
 'brand_name_en': 'Supgyeol',
 'name_meaning': '햇빛에 비친 물결처럼 반짝이는 일상의 창가',
 'slogan': '마음이 쉬어가는 소품의 온도',
 'story_summary': '환경을 생각하는 소비가 개인의 자부심이 되도록, 지속 가능한 소재의 아름다움을 제안합니다.',
 'seed_color': '#F5E6D3',
 'seed_color_reason': '리넨의 내추럴함과 햇살의 포근한 무드'}

interview_data_A = "Q1. 숲결이 제안하는 첫 번째 '소품의 온도'는 어떤 소재를 통해 전달되는 것이 브랜드 스토리와 가장 잘 어울릴까요?\nA: 폐플라스틱이나 폐유리를 투명하고 영롱하게 재탄생시킨 업사이클링 리빙 소품\n\nQ2. 로고와 타이포그래피를 통해 전달하고 싶은 숲결의 '시각적 첫인상'은 무엇인가요?\nA: 우아한 세리프(명조) 스타일: 햇살의 반짝임과 고전적인 미감을 강조한 섬세한 느낌\n\nQ3. 인스타그램 등 주요 소통 채널에서 숲결이 고객에게 말을 건네는 페르소나는 어떤 모습인가요?\nA: 신뢰감 있는 큐레이터: 소재의 가치와 지속 가능한 삶의 방식을 조근조근 설명해 주는 전문적인 대화\n\nQ4. 고객이 택배 상자를 열었을 때, '지속 가능한 자부심'을 느끼게 할 결정적인 한 끗은 무엇이어야 할까요?\nA: 정보적 경험: 제품을 만든 장인이나 소재의 여정이 담긴 감성적인 미니 북렛"
interview_data_B = "Q1. 숲결의 페르소나가 평일 일과를 마치고 현관문을 열었을 때, 가장 먼저 해소하고 싶어 하는 감각적 '결핍'은 무엇인가요?\nA: 자아의 회복: 사회적 역할에서 벗어나 오직 '나의 취항'으로만 채워진 안식\n\nQ2. 이 고객이 숲결의 제품을 구매하며 느끼고 싶어 하는 '가장 결정적인 심리적 트리거'는 무엇인가요?\nA: 취향의 차별화: 흔하지 않은 감도의 브랜드를 발견하고 향유하는 미적 우월감\n\nQ3. 이 타겟이 숲결이라는 브랜드를 처음 발견하게 될 가능성이 가장 높은 경로는 무엇인가요?\nA: 알고리즘 기반 이미지 탐색: 인스타그램이나 핀터레스트의 감각적인 인테리어 무드보드"
interview_data_C = "Q1. 첫 번째 질문: 브랜드의 첫인상을 결정짓는 전체적인 형태와 레이아웃은 어떤 느낌을 지향하시나요?\nA: 광활한 여백을 활용하여 숨통이 트이는 듯한 극도의 미니멀리즘 레이아웃\n\nQ2. 두 번째 질문: 브랜드의 목소리가 되어줄 글꼴(Typography)은 어떤 표정을 짓고 있어야 할까요?\nA: 전통적인 붓 터치의 질감이 살아있어 깊이감과 예스러움이 느껴지는 서예 스타일\n\nQ3. 세 번째 질문: 제품과 브랜드를 담아낼 사진의 '빛과 공기감'은 어떤 온도를 머금고 있어야 할까요?\nA: 안개가 낀 새벽의 숲처럼 차분하고 명상적인 분위기를 주는 낮은 채도의 차분한 톤"

data_C = {'color_palette': [{'color_name': '심연의 숲',   'hex_code': '#2C332B',   'role': 'Text Color',   'reason': '단단하고 정직한 브랜드의 목소리를 대변하며 가독성을 확보하는 깊은 먹색'},  {'color_name': '새벽녘 모래',   'hex_code': '#F5E6D3',   'role': 'Primary',   'reason': '사용자 선택 시드 컬러로, 마음이 쉬어가는 따뜻한 소품의 온도를 상징함'},  {'color_name': '안개 낀 잎새',   'hex_code': '#9BA89B',   'role': 'Secondary',   'reason': '몽환적인 새벽 숲의 분위기를 자아내며 메인 컬러와 차분한 조화를 이룸'},  {'color_name': '마른 나무줄기',   'hex_code': '#8C7B6C',   'role': 'Accent',   'reason': '지속 가능한 소재의 견고함과 자연의 생명력을 강조하기 위한 포인트 컬러'},  {'color_name': '안식의 여백',   'hex_code': '#F9F6F2',   'role': 'Background',   'reason': '시드 컬러보다 밝은 톤으로, 공간감을 확장하고 시각적 피로도를 낮춤'}], 'typography': {'primary_font': {'category': '헤드라인 및 타이틀',   'font_name_kr': '고딕 A1',   'google_fonts_family': 'Gothic A1',   'weight_recommendation': 800,   'usage_and_reason': '획이 굵고 정직한 직선 위주의 서체로, 브랜드의 단단한 철학과 자부심을 시각적으로 전달함'},  'secondary_font': {'category': '본문 및 일반 텍스트',   'font_name_kr': '본고딕',   'google_fonts_family': 'Noto Sans KR',   'weight_recommendation': 400,   'usage_and_reason': '정보 전달의 명확성을 위해 장식성을 배제하고 현대적인 가독성을 극대화함'}}, 'visual_mood_guide': {'photography_do': ['조명 온도를 5000K 이상의 쿨톤으로 설정하고 가습기나 포그 머신을 활용해 미세한 공기감을 연출할 것',   '피사체를 중앙에서 벗어나게 배치하여 여백의 미를 강조하고 자연스러운 시선의 흐름을 유도할 것',   '소재의 질감이 드러나도록 측면에서 들어오는 부드러운 확산광(Diffused Light)을 사용할 것'],  'photography_dont': ['직사광선에 의한 강한 그림자나 인위적인 하이라이트 표현 금지',   '채도가 높은 원색 소품이나 플라스틱 재질의 배경지 사용 금지',   '과도한 광각 렌즈 사용으로 인한 사물의 형태 왜곡 금지'],  'mood_keywords': ['정적인', '정직한', '몽환적인', '유연한', '지속가능한']}, 'design_principles': {'layout_direction': '정형화된 그리드에서 벗어나 유기적인 곡선형 레이아웃을 사용하며, 요소 간 간격을 넓게 유지해 숲길의 여유로운 공간감을 구현합니다. 버튼이나 카드 UI의 모서리는 비정형적인 라운드 값을 적용해 자연물의 형태를 모방합니다.',  'image_processing': '이미지 전체의 채도를 15-20% 낮추고 대비를 완만하게 조절하여 안개 낀 듯한 몽환적인 톤을 유지합니다. 텍스처는 매끄러운 광택보다는 종이나 나무의 거친 질감이 미세하게 느껴지도록 노이즈 값을 소량 추가합니다.'}}


In [ ]:
raw_text_DE_0 = bd.request_gemini_api(pn.get_DE_interview_prompt(brand_info, interview_data_C), SYSTEM_PROMPT)
parsed_response_DE_0 = bd.parse_ai_response(raw_text_DE_0)


----------------------------------------------------------------------------------------------------
💬 AI에 요청 중입니다... 잠시만 기다려주세요.
----------------------------------------------------------------------------------------------------

⚠️ 서버 과부하 또는 할당량 초과 에러 발생: 503 UNAVAILABLE
⏳ 1분 30초 대기 후 재요청합니다... (재시도 횟수: 1/10)


In [ ]:
# AI가 왜 이런 질문을 만들었는지 사용자에게 보여주면 신뢰도가 확 올라갑니다!
print(f"💡 AI 분석: {parsed_response_DE_0['reasoning']}\n")

# 사용자의 최종 답변을 모아둘 리스트
collected_answers_DE = bd.conduct_ai_interview(parsed_response_DE_0, title="🤖 Section D 비주얼 아이덴티티 인터뷰")
interview_data_DE = bd.format_interview_responses(collected_answers_DE)

💡 AI 분석: 다음 단계에서 [로고 및 캐릭터 기획안]을 구체적으로 작성하기 위해, 현재 심볼의 구체적인 조형적 모티프, 질감의 밀도, 캐릭터의 생명체 유형 및 성격에 대한 정보가 부족하므로 이를 파악할 질문 4개를 생성했습니다.


🤖 Section D 비주얼 아이덴티티 인터뷰
💡 AI 분석: 다음 단계에서 [로고 및 캐릭터 기획안]을 구체적으로 작성하기 위해, 현재 심볼의 구체적인 조형적 모티프, 질감의 밀도, 캐릭터의 생명체 유형 및 성격에 대한 정보가 부족하므로 이를 파악할 질문 4개를 생성했습니다.


[질문 1/4. 브랜드의 핵심 메타포인 '물결'과 '숲'을 로고 심볼로 형상화한다면, 어떤 구체적인 조형적 이미지가 가장 브랜드의 본질에 가깝나요?]
  1. 잔잔한 수면 위에 번지는 동심원을 극도로 단순화한 미니멀 선형 메타포
  2. 안개 낀 숲의 나무 실루엣을 한 획의 붓 터치로 표현한 여백 중심의 회화적 메타포
  3. 창가로 스며드는 빛의 줄기와 그림자를 기하학적 면으로 분할한 현대적 추상 메타포
  4. 한글 '숲'의 자음과 모음을 나뭇가지와 잎사귀 형상으로 재해석한 유기적 타이포그래피 메타포

[질문 2/4. 심볼의 선(Line)이 전달하는 시각적 질감과 디테일은 어떤 해상도를 지향하시나요?]
  1. 잉크가 번진 듯 끝처리가 부드럽고 스며드는 느낌이 강한 수묵화 스타일
  2. 칼로 깎아낸 듯 날카롭고 힘 있는 질감이 느껴지는 투박한 목판화 스타일
  3. 현대적이고 도시적인 느낌을 주는 일정하고 매우 가느다란 굵기의 라인 아트 스타일
  4. 거친 종이 위에 연필로 슥슥 그려낸 듯한 따뜻하고 아날로그적인 핸드 드로잉 스타일

[질문 3/4. 안개 낀 새벽 숲의 평온함을 대변할 브랜드 캐릭터로 어떤 존재가 가장 직관적으로 떠오르시나요?]
  1. 숲의 안개를 먹고 자라며 형체가 자유자재로 변하는 몽글몽글한 '빛의 정령'
  2. 느릿한 걸음으로 숲을 산책하며 소품을 수집하는 긴 뿔을 가진 우아한 '사슴'
  3. 세월의 흔적을 간직한 채

In [ ]:
raw_text_DE_1 = bd.request_gemini_api(pn.get_DE_identity_prompt(brand_info, data_C, f"{interview_data_A} + {interview_data_B} + {interview_data_C}", interview_data_DE), SYSTEM_PROMPT)
parsed_response_DE_1 = bd.parse_ai_response(raw_text_DE_1)


----------------------------------------------------------------------------------------------------
💬 AI에 요청 중입니다... 잠시만 기다려주세요.
----------------------------------------------------------------------------------------------------

⚠️ 서버 과부하 또는 할당량 초과 에러 발생: 429 RESOURCE_EXHAUSTED
⏳ 1분 30초 대기 후 재요청합니다... (재시도 횟수: 1/10)


In [ ]:
logo_path = bd.generate_logo_image(brand_info['brand_name'], parsed_response_DE_1, GEMINI_IMAGE_API_KEY)
char_path = bd.generate_character_image(brand_info['brand_name'], parsed_response_DE_1, GEMINI_IMAGE_API_KEY)


----------------------------------------------------------------------------------------------------
🎨 AI 이미지 생성 요청 중입니다... 잠시만 기다려주세요.
----------------------------------------------------------------------------------------------------


In [ ]:
parsed_response_DE_1['logo_path'] = logo_path
parsed_response_DE_1['char_path'] = char_path

NameError: name 'logo_path' is not defined

In [ ]:
print(parsed_response_DE_1)

NameError: name 'parsed_response_DE_1' is not defined